# 04 — A/B Testing

Now we "run" the experiment we designed in Notebook 2, at the sample size we planned in Notebook 3,
and walk through the full analysis pipeline:

**Control mean → Treatment mean → Difference → Variance → Standard Error → Confidence Interval →
Welch's t-test → p-value → Business recommendation**


In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_processing import BusinessConfig, generate_synthetic_customers
from experiment import TreatmentConfig, run_experiment, summarize_groups
from stats_tools import welch_t_test, required_sample_size, cohens_d

plt.style.use("seaborn-v0_8-whitegrid")


## 1. Generate the population and run the experiment at our planned sample size

In [ ]:
# From Notebook 3: ~7,080 customers/group needed. We generate a population
# large enough to comfortably support that, then run the experiment.
population_config = BusinessConfig(n_customers=20_000, seed=42)
customers = generate_synthetic_customers(population_config)

treatment_config = TreatmentConfig(price_increase_pct=0.10, elasticity=0.5, seed=123)
experiment_df = run_experiment(customers, treatment_config)

experiment_df.groupby("group").size()


## 2. Control mean, Treatment mean, Difference

In [ ]:
control_rev = experiment_df.loc[experiment_df["group"] == "control", "revenue"].to_numpy()
treatment_rev = experiment_df.loc[experiment_df["group"] == "treatment", "revenue"].to_numpy()

control_mean = control_rev.mean()
treatment_mean = treatment_rev.mean()
diff = treatment_mean - control_mean

print(f"Control mean revenue/customer:   ${control_mean:,.2f}  (n={len(control_rev):,})")
print(f"Treatment mean revenue/customer: ${treatment_mean:,.2f}  (n={len(treatment_rev):,})")
print(f"Difference (Treatment - Control): ${diff:,.2f}  ({diff/control_mean:+.2%})")


## 3. Variance → Standard Error → Confidence Interval → Welch's t-test → p-value

In [ ]:
result = welch_t_test(control_rev, treatment_rev, alpha=0.05)
result


In [ ]:
print(f"Control variance:      {control_rev.var(ddof=1):,.2f}")
print(f"Treatment variance:    {treatment_rev.var(ddof=1):,.2f}")
print(f"Standard error (diff): {result.se_diff:,.3f}")
print(f"95% CI on difference:  [${result.ci_low:,.2f}, ${result.ci_high:,.2f}]")
print(f"t-statistic:           {result.t_stat:,.3f}")
print(f"p-value:               {result.p_value:.4f}")
print()
print(f"Statistically significant at alpha=0.05? {result.significant}")
print(f"RECOMMENDATION: {result.recommendation}")


## 4. Visualize the comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(control_rev, bins=60, alpha=0.6, label="Control", color="#4C72B0")
axes[0].hist(treatment_rev, bins=60, alpha=0.6, label="Treatment", color="#C44E52")
axes[0].axvline(control_mean, color="#4C72B0", linestyle="--")
axes[0].axvline(treatment_mean, color="#C44E52", linestyle="--")
axes[0].set_title("Revenue per customer: Control vs Treatment")
axes[0].set_xlabel("Revenue ($)")
axes[0].legend()

axes[1].errorbar(
    x=[0, 1],
    y=[control_mean, treatment_mean],
    yerr=[
        control_rev.std(ddof=1) / np.sqrt(len(control_rev)) * 1.96,
        treatment_rev.std(ddof=1) / np.sqrt(len(treatment_rev)) * 1.96,
    ],
    fmt="o",
    capsize=8,
    markersize=10,
    color="black",
)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Control", "Treatment"])
axes[1].set_ylabel("Mean revenue/customer ($)")
axes[1].set_title("Group means with 95% CI")

plt.tight_layout()
plt.show()


## 5. Guardrail metrics

A primary-metric win doesn't automatically mean "ship it" — we check the guardrails from Notebook 1.

In [ ]:
guardrails = summarize_groups(experiment_df)
guardrails


### Reading the guardrails

- **conversion_rate**: did fewer customers purchase at all under the higher price?
- **avg_orders**: are existing customers buying less often (the elasticity effect in action)?
- **avg_order_value**: is the ~10% price increase actually landing in realized order value?

If revenue per customer is up but conversion has collapsed, that's a signal the win is fragile /
short-term (existing customers loading up before leaving, for example) and worth a longer look
before rolling out permanently.

## 6. Business recommendation

Bringing it together:

1. Revenue per customer moved by the amount and direction shown above, and the 95% CI/​p-value tell
   us how much to trust that it isn't just noise.
2. Guardrails (conversion, orders, order value) tell us *why* it moved, and whether that's healthy.
3. Combine both before recommending: **see the printed `result.recommendation` above**

## Next: `05_sensitivity_analysis.ipynb` — would this recommendation survive if our elasticity
assumption were wrong?